In [ ]:
import numpy as np

In [ ]:
class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None

In [ ]:
class DecisionTreeRegressor:
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        self.root = self._grow_tree(X, y, depth=0)

    def predict(self, X):
        X = np.array(X, dtype=float)
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape

        if depth >= self.max_depth or n_samples < self.min_samples_split or len(np.unique(y)) == 1:
            return TreeNode(value=np.mean(y))

        best_feature, best_threshold = self._best_split(X, y)

        if best_feature is None:
            return TreeNode(value=np.mean(y))

        left_idxs, right_idxs = self._split(X[:, best_feature], best_threshold)

        left = self._grow_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs], y[right_idxs], depth + 1)

        return TreeNode(feature=best_feature, threshold=best_threshold, left=left, right=right)

    def _best_split(self, X, y):
        best_mse = float("inf")
        split_idx = None
        split_threshold = None

        n_features = X.shape[1]

        for feature_idx in range(n_features):
            X_column = X[:, feature_idx]
            thresholds = self._candidate_thresholds(X_column)

            for threshold in thresholds:
                left_idxs, right_idxs = self._split(X_column, threshold)

                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue

                mse = self._split_mse(y, left_idxs, right_idxs)

                if mse < best_mse:
                    best_mse = mse
                    split_idx = feature_idx
                    split_threshold = threshold

        return split_idx, split_threshold

    def _candidate_thresholds(self, X_column):
        values = np.unique(X_column)
        if len(values) <= 1:
            return values
        return (values[:-1] + values[1:]) / 2

    def _split(self, X_column, threshold):
        left_idxs = np.argwhere(X_column <= threshold).flatten()
        right_idxs = np.argwhere(X_column > threshold).flatten()
        return left_idxs, right_idxs

    def _split_mse(self, y, left_idxs, right_idxs):
        y_left = y[left_idxs]
        y_right = y[right_idxs]

        left_mse = np.mean((y_left - np.mean(y_left)) ** 2) if len(y_left) > 0 else 0
        right_mse = np.mean((y_right - np.mean(y_right)) ** 2) if len(y_right) > 0 else 0

        n = len(y)
        return (len(y_left) / n) * left_mse + (len(y_right) / n) * right_mse

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [ ]:
class GradientBoostingRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []
        self.initial_prediction = None

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)

        self.initial_prediction = np.mean(y)
        y_pred = np.full(len(y), self.initial_prediction, dtype=float)
        self.trees = []

        for _ in range(self.n_estimators):
            residuals = y - y_pred

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split
            )

            tree.fit(X, residuals)
            update = tree.predict(X)

            y_pred += self.learning_rate * update
            self.trees.append(tree)

    def predict(self, X):
        X = np.array(X, dtype=float)
        y_pred = np.full(X.shape[0], self.initial_prediction, dtype=float)

        for tree in self.trees:
            y_pred += self.learning_rate * tree.predict(X)

        return y_pred

    def mse(self, X, y):
        y = np.array(y, dtype=float)
        y_pred = self.predict(X)
        return np.mean((y - y_pred) ** 2)

    def rmse(self, X, y):
        return np.sqrt(self.mse(X, y))

    def r2_score(self, X, y):
        y = np.array(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

In [ ]:
X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [6],
    [7],
    [8]
], dtype=float)

y = np.array([1.2, 1.9, 3.2, 3.9, 5.1, 5.8, 7.2, 7.9], dtype=float)

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=2,
    min_samples_split=2
)

gbr.fit(X, y)

predictions = gbr.predict(X)

print("Predictions:", predictions)
print("MSE:", gbr.mse(X, y))
print("RMSE:", gbr.rmse(X, y))
print("R2:", gbr.r2_score(X, y))

In [ ]:
X_new = np.array([
    [2.5],
    [4.5],
    [6.5],
    [9.0]
], dtype=float)

print(gbr.predict(X_new))

In [ ]:
def print_tree(node, depth=0):
    prefix = "  " * depth
    if node.is_leaf():
        print(prefix + f"Leaf: {node.value}")
        return
    print(prefix + f"X[{node.feature}] <= {node.threshold}")
    print_tree(node.left, depth + 1)
    print_tree(node.right, depth + 1)

In [ ]:
print_tree(gbr.trees[0].root)